# 01 - Preparar paths destino para datastore

Primera etapa del flujo Geosupport. Este notebook recibe una carpeta raiz con imagenes nuevas, calcula el sector por cruce espacial contra el indice de vuelos, arma el nombre oficial y define el `Path_Destino` en el datastore.

No copia archivos, no carga al mosaico y no modifica datos. El resultado principal para revisar es `04_ready_for_datastore.csv`.

In [1]:
from datetime import datetime
from pathlib import Path
import importlib
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'core').exists():
    for candidate in [Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / 'core').exists():
            PROJECT_ROOT = candidate
            break

if not (PROJECT_ROOT / 'core').exists():
    raise FileNotFoundError('No se encontro el folder core. Ejecuta el notebook desde la raiz del proyecto o desde flujo_geosupport_etapas.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

FLOW_DIR = PROJECT_ROOT / 'flujo_geosupport_etapas'

import core.mosaic_image_audit as mosaic_audit
mosaic_audit = importlib.reload(mosaic_audit)
from core.mosaic_image_audit import *

print('Proyecto raiz:', PROJECT_ROOT)
print('Folder flujo:', FLOW_DIR)
print('Modulo auditoria:', mosaic_audit.__file__)
print('Version logica:', AUDIT_LOGIC_VERSION)

Proyecto raiz: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport
Folder flujo: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\flujo_geosupport_etapas
Modulo auditoria: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\core\mosaic_image_audit.py
Version logica: 2026-06-15-spatial-sector-preserve-token


## Parametros

Cambiar solo `PATH_INPUT_IMAGENES` para una nueva entrega. El resto deja fija la logica validada del proyecto.

In [2]:
# Carpeta raiz donde llegan las imagenes nuevas. La busqueda es recursiva.
PATH_INPUT_IMAGENES = r"\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_Drone_Sin_Procesar\INPUT\20260206_Geosupport_primera entrega"

# Feature class con los footprints/sectores usados para definir el nombre geografico.
PATH_FC_INDICE_VUELOS_IMGS = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb\CL_MLP_PAO_06_COMPLEMENTOS\CL_MLP_PAO_Indice_Vuelos_PAO_IMGS_PO"
SECTOR_FIELD_INDICE_VUELOS = "Sector"
QUERY_INDICE_VUELOS = "Sensor <> 'DJI MATRICE 350 RTK'"

# Raiz del datastore donde quedaran las imagenes copiadas en la etapa siguiente.
PATH_DATASTORE_DESTINO_RAIZ = r"\\amssclgis10.ams.gmams.cl\CL_MLP_PAO"

# Salida local de auditoria/manifiesto.
run_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = FLOW_DIR / 'outputs' / 'etapa_01_preparar_paths_datastore'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Input imagenes:', PATH_INPUT_IMAGENES)
print('Feature sectores:', PATH_FC_INDICE_VUELOS_IMGS)
print('Destino datastore raiz:', PATH_DATASTORE_DESTINO_RAIZ)
print('Campo sector:', SECTOR_FIELD_INDICE_VUELOS)
print('Query sectores:', QUERY_INDICE_VUELOS)
print('Salida:', OUTPUT_DIR)

Input imagenes: \\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_Drone_Sin_Procesar\INPUT\20260206_Geosupport_primera entrega
Feature sectores: \\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb\CL_MLP_PAO_06_COMPLEMENTOS\CL_MLP_PAO_Indice_Vuelos_PAO_IMGS_PO
Destino datastore raiz: \\amssclgis10.ams.gmams.cl\CL_MLP_PAO
Campo sector: Sector
Query sectores: Sensor <> 'DJI MATRICE 350 RTK'
Salida: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\flujo_geosupport_etapas\outputs\etapa_01_preparar_paths_datastore


## Reglas de carpeta destino

El nombre de archivo se define por fecha + sector geografico. La carpeta destino se define por sector. Cuando un archivo puntual necesita una excepcion, se resuelve primero por nombre original.

In [3]:
DESTINATION_FOLDER_BY_SECTOR = {
    'Camino_Alternativo_Salamanca': 'El_Mauro_Drone',
    'DME9-PA12-IIFF8': 'Chacay_El_Mauro_Drone',
    'EBD': 'El_Mauro_Drone',
    'ED1': 'Chacay_El_Mauro_Drone',
    'ED2': 'Chacay_El_Mauro_Drone',
    'EDT': 'Puerto_Punta_Chungo_Drone',
    'EM1': 'Chacay_Drone',
    'EM2_S2': 'Chacay_El_Mauro_Drone',
    'EM3': 'Chacay_El_Mauro_Drone',
    'Estacion_Cabecera': 'Chacay_Drone',
    'Estacion_Intermedia': 'Chacay_El_Mauro_Drone',
    'EV1': 'Chacay_El_Mauro_Drone',
    'EV2': 'El_Mauro_Puerto_Punta_Chungo_Drone',
    'Helipuerto': 'El_Mauro_Drone',
    'MonteAranda-NSTC-Km-84p2-a-82p3': 'Chacay_El_Mauro_Drone',
    'Patio-19B-y-Armado': 'Chacay_El_Mauro_Drone',
    'Subestacion-El-Mauro': 'El_Mauro_Drone',
    'Subestacion-El-Mauro_A_E35': 'El_Mauro_Drone',
    'TORRE_E85_A_E_125': 'Chacay_El_Mauro_Drone',
    'TORRES_E31_A_E48_PV4': 'Chacay_El_Mauro_Drone',
    'TORRES_E48_A_E84_PV4': 'Chacay_El_Mauro_Drone',
}

DESTINATION_FOLDER_BY_FILE_NAME = {
    'GEOSP-TRN-002603_ORTOFOTO_CORTADA_EM2_100526.tif': 'Chacay_El_Mauro_Drone',
    'GEOSP-TRN-002615_GS_ORTOFOTO_ESTACION DE MONITOREO NÂ°2_13-05-2026.tif': 'El_Mauro_Drone',
    'GEOSP-TRN-002617_GS_ORTOFOTO_SUBESTACION_EL MAURO_PRIORIDAD 1_13_05_26.tif': 'Chacay_El_Mauro_Drone',
    'GEOSP-TRN-002545_GS_ORTOFOTO_EB3_06-05-26.tif': 'El_Mauro_Drone',
    'GEOSP-TRN-002546_GS_ORTOFOTO_SSEE_06-05-26.tif': 'El_Mauro_Drone',
    'GEOSP-TRN-002621_GS_ORTOFOTO_ESTACION DE BOMBEO NÂº3_13_05_26.tif': 'El_Mauro_Drone',
}

display(pd.DataFrame(
    [{'Sector': key, 'Carpeta_Destino': value} for key, value in DESTINATION_FOLDER_BY_SECTOR.items()]
).sort_values('Sector').reset_index(drop=True))

,Sector,Carpeta_Destino
0,Camino_Alternativo_Salamanca,El_Mauro_Drone
1,DME9-PA12-IIFF8,Chacay_El_Mauro_Drone
2,EBD,El_Mauro_Drone
3,ED1,Chacay_El_Mauro_Drone
4,ED2,Chacay_El_Mauro_Drone
5,EDT,Puerto_Punta_Chungo_Drone
6,EM1,Chacay_Drone
7,EM2_S2,Chacay_El_Mauro_Drone
8,EM3,Chacay_El_Mauro_Drone
9,EV1,Chacay_El_Mauro_Drone


## 1. Buscar imagenes

La busqueda es recursiva. Para el mosaico solo se preparan archivos `tif` y `tiff`.

In [4]:
input_images_df = scan_input_images(PATH_INPUT_IMAGENES)
ortho_images_df = input_images_df[input_images_df['extension'].isin(ORTHO_MOSAIC_EXTENSIONS)].copy()

print(f'Archivos encontrados: {len(input_images_df)}')
print(f'Imagenes TIF/TIFF para evaluar: {len(ortho_images_df)}')

display(input_images_df.groupby('extension').size().reset_index(name='count'))
display(ortho_images_df[['file_name', 'relative_path', 'size_mb', 'modified_at']].head(20))

Archivos encontrados: 607
Imagenes TIF/TIFF para evaluar: 33


,extension,count
0,.jpg,574
1,.tif,33


,file_name,relative_path,size_mb,modified_at
574,GEOSP-TRN-001426_GS_ORTOFOTO EB2 28-12-2025.tif,SOLO_TIF_02Jun26\GEOSP-TRN-001426_GS_ORTOFOTO ...,110.944,2026-06-11 10:37:57
575,GEOSP-TRN-001492_GS_ORTOFOTO_Estacion de bombe...,SOLO_TIF_02Jun26\GEOSP-TRN-001492_GS_ORTOFOTO_...,155.039,2026-06-11 10:38:02
576,GEOSP-TRN-001553_GS_ORTOFOTO_EB2_20-12-2025.tif,SOLO_TIF_02Jun26\GEOSP-TRN-001553_GS_ORTOFOTO_...,147.589,2026-06-11 10:38:05
577,GEOSP-TRN-001642_ORTOFOTO_EB2_03-01-2026.tif,SOLO_TIF_02Jun26\GEOSP-TRN-001642_ORTOFOTO_EB2...,180.745,2026-06-11 10:38:07
578,GEOSP-TRN-001732_GS_ORTOFOTO_EB2_17-01-26.tif,SOLO_TIF_02Jun26\GEOSP-TRN-001732_GS_ORTOFOTO_...,164.324,2026-06-11 10:38:09
579,GEOSP-TRN-001784_GS_ORTOFOTO_Estacion de bombe...,SOLO_TIF_02Jun26\GEOSP-TRN-001784_GS_ORTOFOTO_...,96.611,2026-06-11 10:38:11
580,GEOSP-TRN-001868_GS_ORTOFOTO EB2 04-02-2026.tif,SOLO_TIF_02Jun26\GEOSP-TRN-001868_GS_ORTOFOTO ...,128.088,2026-06-11 10:38:15
581,GEOSP-TRN-001975_GS_ORTOFOTO_Estacion de bombe...,SOLO_TIF_02Jun26\GEOSP-TRN-001975_GS_ORTOFOTO_...,147.604,2026-06-11 10:38:18
582,GEOSP-TRN-002016_GS_ORTOFOTO_EB2_26-02-26.tif,SOLO_TIF_02Jun26\GEOSP-TRN-002016_GS_ORTOFOTO_...,39.182,2026-06-11 10:38:21
583,GEOSP-TRN-002096_GS_ORTOFOTO_EB2_07-03-2026.tif,SOLO_TIF_02Jun26\GEOSP-TRN-002096_GS_ORTOFOTO_...,225.507,2026-06-11 10:38:24


## 2. Calcular sector geografico y nombre esperado

El sector viene solo del cruce espacial. Si una imagen cruza mas de un sector, se usa el sector con mayor porcentaje de interseccion. El texto del sector se conserva desde el feature class y solo se reemplazan espacios por `_`.

In [5]:
spatial_matches_df = calculate_spatial_sector_matches(
    ortho_images_df,
    PATH_FC_INDICE_VUELOS_IMGS,
    sector_field=SECTOR_FIELD_INDICE_VUELOS,
    where_clause=QUERY_INDICE_VUELOS,
)

prepared_df = add_expected_names_with_spatial_sector(
    ortho_images_df,
    spatial_matches_df,
)

display(spatial_matches_df['spatial_status'].value_counts(dropna=False).reset_index(name='count').rename(columns={'index': 'spatial_status'}))
display(prepared_df['rename_status'].value_counts(dropna=False).reset_index(name='count').rename(columns={'index': 'rename_status'}))

,spatial_status,count
0,ok,25
1,sin_cruce_sector,8


,rename_status,count
0,ok,16
1,descartar_posible_plano,9
2,sin_fecha,6
3,sin_cruce_sector,2


## 3. Preparar manifest de paths destino

`ready_for_datastore` indica imagenes con nombre y destino completos. Si hay nombres repetidos, se agrega una secuencia `-1`, `-2`, etc. al final del nombre.

In [6]:
manifest_df = resolve_duplicate_expected_names(prepared_df)

manifest_df['destination_date_folder'] = manifest_df['expected_date_token'].map(
    lambda value: '_'.join(str(value).split('_')[:2]) if pd.notna(value) and value else None
)

destination_folder_lookup = {normalize_key(key): value for key, value in DESTINATION_FOLDER_BY_SECTOR.items()}
destination_file_lookup = {normalize_key(key): value for key, value in DESTINATION_FOLDER_BY_FILE_NAME.items()}

def resolve_destination_folder(row):
    file_folder = destination_file_lookup.get(normalize_key(row.get('file_name')))
    if file_folder:
        return file_folder
    return destination_folder_lookup.get(normalize_key(row.get('expected_sector')))

def build_destination_path(row):
    if pd.isna(row.get('destination_folder')) or pd.isna(row.get('destination_date_folder')) or pd.isna(row.get('expected_file_name')):
        return None
    return str(Path(PATH_DATASTORE_DESTINO_RAIZ) / str(row['destination_folder']) / str(row['destination_date_folder']) / str(row['expected_file_name']))

def build_review_reason(row):
    reasons = []
    if row.get('rename_status') != 'ok':
        reasons.append(str(row.get('rename_status')))
    if row.get('rename_status') == 'ok' and not row.get('destination_folder'):
        reasons.append('sin_regla_carpeta_destino')
    if row.get('duplicate_was_resolved'):
        reasons.append('nombre_duplicado_resuelto')
    if row.get('spatial_overlap_count', 0) and row.get('spatial_overlap_count', 0) > 1:
        reasons.append('cruza_multiples_sectores')
    return '|'.join(reasons) if reasons else None

manifest_df['destination_folder'] = manifest_df.apply(resolve_destination_folder, axis=1)
manifest_df['destination_path'] = manifest_df.apply(build_destination_path, axis=1)
manifest_df['ready_for_datastore'] = manifest_df['rename_status'].eq('ok') & manifest_df['destination_path'].notna()
manifest_df['review_reason'] = manifest_df.apply(build_review_reason, axis=1)

output_columns = [
    'ready_for_datastore',
    'review_reason',
    'path',
    'relative_path',
    'file_name',
    'expected_file_name',
    'destination_path',
    'original_expected_file_name',
    'expected_name',
    'expected_date_token',
    'destination_folder',
    'destination_date_folder',
    'expected_sector',
    'sector_source',
    'rename_status',
    'spatial_status',
    'spatial_sector_raw',
    'spatial_sector',
    'spatial_overlap_pct',
    'spatial_overlap_count',
    'spatial_all_matches',
    'duplicate_expected_file_name',
    'duplicate_sequence',
    'duplicate_was_resolved',
    'size_mb',
    'modified_at',
]
output_columns = [column for column in output_columns if column in manifest_df.columns]
manifest_output_df = manifest_df[output_columns].copy()

ready_df = manifest_output_df[manifest_output_df['ready_for_datastore']].copy()
review_df = manifest_output_df[~manifest_output_df['ready_for_datastore']].copy()

print(f'Listas para validar/copiar: {len(ready_df)}')
print(f'Requieren revision: {len(review_df)}')
print(f'Nombres duplicados resueltos con secuencia: {int(manifest_output_df["duplicate_was_resolved"].sum())}')

display(manifest_output_df.head(30))
display(review_df.head(30))

Listas para validar/copiar: 16
Requieren revision: 17
Nombres duplicados resueltos con secuencia: 0


,ready_for_datastore,review_reason,path,relative_path,file_name,expected_file_name,destination_path,original_expected_file_name,expected_name,expected_date_token,...,spatial_sector_raw,spatial_sector,spatial_overlap_pct,spatial_overlap_count,spatial_all_matches,duplicate_expected_file_name,duplicate_sequence,duplicate_was_resolved,size_mb,modified_at
0,True,cruza_multiples_sectores,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_02Jun26\GEOSP-TRN-001426_GS_ORTOFOTO ...,GEOSP-TRN-001426_GS_ORTOFOTO EB2 28-12-2025.tif,CL_MLP_PAO_IF_Ortho_25_12_28_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,CL_MLP_PAO_IF_Ortho_25_12_28_MonteAranda-NSTC-...,CL_MLP_PAO_IF_Ortho_25_12_28_MonteAranda-NSTC-...,25_12_28,...,MonteAranda-NSTC-Km-84p2-a-82p3,MonteAranda-NSTC-Km-84p2-a-82p3,100.000023,18,MonteAranda-NSTC-Km-84p2-a-82p3:100.00|MonteAr...,False,<NA>,False,110.944,2026-06-11 10:37:57
1,False,sin_fecha|cruza_multiples_sectores,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_02Jun26\GEOSP-TRN-001492_GS_ORTOFOTO_...,GEOSP-TRN-001492_GS_ORTOFOTO_Estacion de bombe...,None,None,None,None,None,...,MonteAranda-NSTC-Km-84p2-a-82p3,MonteAranda-NSTC-Km-84p2-a-82p3,99.999957,18,MonteAranda-NSTC-Km-84p2-a-82p3:100.00|MonteAr...,False,<NA>,False,155.039,2026-06-11 10:38:02
2,True,cruza_multiples_sectores,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_02Jun26\GEOSP-TRN-001553_GS_ORTOFOTO_...,GEOSP-TRN-001553_GS_ORTOFOTO_EB2_20-12-2025.tif,CL_MLP_PAO_IF_Ortho_25_12_20_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,CL_MLP_PAO_IF_Ortho_25_12_20_MonteAranda-NSTC-...,CL_MLP_PAO_IF_Ortho_25_12_20_MonteAranda-NSTC-...,25_12_20,...,MonteAranda-NSTC-Km-84p2-a-82p3,MonteAranda-NSTC-Km-84p2-a-82p3,100.000010,18,MonteAranda-NSTC-Km-84p2-a-82p3:100.00|MonteAr...,False,<NA>,False,147.589,2026-06-11 10:38:05
3,True,cruza_multiples_sectores,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_02Jun26\GEOSP-TRN-001642_ORTOFOTO_EB2...,GEOSP-TRN-001642_ORTOFOTO_EB2_03-01-2026.tif,CL_MLP_PAO_IF_Ortho_26_01_03_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,CL_MLP_PAO_IF_Ortho_26_01_03_MonteAranda-NSTC-...,CL_MLP_PAO_IF_Ortho_26_01_03_MonteAranda-NSTC-...,26_01_03,...,MonteAranda-NSTC-Km-84p2-a-82p3,MonteAranda-NSTC-Km-84p2-a-82p3,99.999970,18,MonteAranda-NSTC-Km-84p2-a-82p3:100.00|MonteAr...,False,<NA>,False,180.745,2026-06-11 10:38:07
4,True,cruza_multiples_sectores,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_02Jun26\GEOSP-TRN-001732_GS_ORTOFOTO_...,GEOSP-TRN-001732_GS_ORTOFOTO_EB2_17-01-26.tif,CL_MLP_PAO_IF_Ortho_26_01_17_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,CL_MLP_PAO_IF_Ortho_26_01_17_MonteAranda-NSTC-...,CL_MLP_PAO_IF_Ortho_26_01_17_MonteAranda-NSTC-...,26_01_17,...,MonteAranda-NSTC-Km-84p2-a-82p3,MonteAranda-NSTC-Km-84p2-a-82p3,100.000023,18,MonteAranda-NSTC-Km-84p2-a-82p3:100.00|MonteAr...,False,<NA>,False,164.324,2026-06-11 10:38:09
5,False,sin_fecha|cruza_multiples_sectores,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_02Jun26\GEOSP-TRN-001784_GS_ORTOFOTO_...,GEOSP-TRN-001784_GS_ORTOFOTO_Estacion de bombe...,None,None,None,None,None,...,MonteAranda-NSTC-Km-84p2-a-82p3,MonteAranda-NSTC-Km-84p2-a-82p3,100.000001,18,MonteAranda-NSTC-Km-84p2-a-82p3:100.00|EB2:100...,False,<NA>,False,96.611,2026-06-11 10:38:11
6,True,cruza_multiples_sectores,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_02Jun26\GEOSP-TRN-001868_GS_ORTOFOTO ...,GEOSP-TRN-001868_GS_ORTOFOTO EB2 04-02-2026.tif,CL_MLP_PAO_IF_Ortho_26_02_04_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,CL_MLP_PAO_IF_Ortho_26_02_04_MonteAranda-NSTC-...,CL_MLP_PAO_IF_Ortho_26_02_04_MonteAranda-NSTC-...,26_02_04,...,MonteAranda-NSTC-Km-84p2-a-82p3,MonteAranda-NSTC-Km-84p2-a-82p3,99.999946,18,MonteAranda-NSTC-Km-84p2-a-82p3:100.00|MonteAr...,False,<NA>,False,128.088,2026-06-11 10:38:15
7,False,sin_fecha|cruza_multiples_se

,ready_for_datastore,review_reason,path,relative_path,file_name,expected_file_name,destination_path,original_expected_file_name,expected_name,expected_date_token,...,spatial_sector_raw,spatial_sector,spatial_overlap_pct,spatial_overlap_count,spatial_all_matches,duplicate_expected_file_name,duplicate_sequence,duplicate_was_resolved,size_mb,modified_at
1,False,sin_fecha|cruza_multiples_sectores,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_02Jun26\GEOSP-TRN-001492_GS_ORTOFOTO_...,GEOSP-TRN-001492_GS_ORTOFOTO_Estacion de bombe...,None,None,None,None,None,...,MonteAranda-NSTC-Km-84p2-a-82p3,MonteAranda-NSTC-Km-84p2-a-82p3,99.999957,18,MonteAranda-NSTC-Km-84p2-a-82p3:100.00|MonteAr...,False,<NA>,False,155.039,2026-06-11 10:38:02
5,False,sin_fecha|cruza_multiples_sectores,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_02Jun26\GEOSP-TRN-001784_GS_ORTOFOTO_...,GEOSP-TRN-001784_GS_ORTOFOTO_Estacion de bombe...,None,None,None,None,None,...,MonteAranda-NSTC-Km-84p2-a-82p3,MonteAranda-NSTC-Km-84p2-a-82p3,100.000001,18,MonteAranda-NSTC-Km-84p2-a-82p3:100.00|EB2:100...,False,<NA>,False,96.611,2026-06-11 10:38:11
7,False,sin_fecha|cruza_multiples_sectores,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_02Jun26\GEOSP-TRN-001975_GS_ORTOFOTO_...,GEOSP-TRN-001975_GS_ORTOFOTO_Estacion de bombe...,None,None,None,None,None,...,MonteAranda-NSTC-Km-84p2-a-82p3,MonteAranda-NSTC-Km-84p2-a-82p3,99.999988,18,MonteAranda-NSTC-Km-84p2-a-82p3:100.00|MonteAr...,False,<NA>,False,147.604,2026-06-11 10:38:18
11,False,sin_fecha|cruza_multiples_sectores,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_02Jun26\GEOSP-TRN-002200_ORTOFOTO COM...,GEOSP-TRN-002200_ORTOFOTO COMPLETA.tif,None,None,None,None,None,...,MonteAranda-NSTC-Km-84p2-a-82p3,MonteAranda-NSTC-Km-84p2-a-82p3,99.999978,18,MonteAranda-NSTC-Km-84p2-a-82p3:100.00|MonteAr...,False,<NA>,False,29.916,2026-06-11 10:38:35
13,False,sin_fecha|cruza_multiples_sectores,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_02Jun26\GEOSP-TRN-002300_ortofoto com...,GEOSP-TRN-002300_ortofoto completa.tif,None,None,None,None,None,...,MonteAranda-NSTC-Km-84p2-a-82p3,MonteAranda-NSTC-Km-84p2-a-82p3,100.000002,18,MonteAranda-NSTC-Km-84p2-a-82p3:100.00|MonteAr...,False,<NA>,False,50.335,2026-06-11 10:38:43
17,False,descartar_posible_plano|cruza_multiples_sectores,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_02Jun26\GEOSP-TRN-002504_1001-03-T-CS...,GEOSP-TRN-002504_1001-03-T-CS-202-4310-C-DW-23...,None,None,None,None,None,...,Subestacion-El-Mauro_A_E35,Subestacion-El-Mauro_A_E35,99.999964,18,Subestacion-El-Mauro_A_E35:100.00|Subestacion-...,False,<NA>,False,41.831,2026-06-11 10:38:54
19,False,descartar_posible_plano,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_02Jun26\GEOSP-TRN-002517_1001-03-T-CS...,GEOSP-TRN-002517_1001-03-T-CS-202-5600-C-DW-11...,None,None,None,None,None,...,None,None,0.000000,0,NaN,False,<NA>,False,128.886,2026-06-11 10:39:03
21,False,descartar_posible_plano|cruza_multiples_sectores,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_02Jun26\GEOSP-TRN-002551_1001-03-T-CS...,GEOSP-TRN-002551_1001-03-T-CS-202-5600-C-DW-11...,None,None,None,None,None,...,TORRE_E85_A_E_125,TORRE_E85_A_E_125,93.006965,16,TORRE_E85_A_E_125:93.01|TORRE_E85_A_E_125:92.9...,False,<NA>,False,37.785,2026-06-11 10:39:07
22,False,descartar_posible_plano,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_02Jun26\GEOSP-TRN-002562_1001-03-T-CS...,GEOSP-TRN-002562_1001-03-T-CS-202-5600-C-DW-11...,None,None,None,None,None,...,None,None,0.000000,0,NaN,False,<NA>,False,47.647,2026-06-11 10:39:08
23,False,descartar_posible_plano|cruza_multiples_sectores,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,SOLO_TIF_02Jun26\GEOSP-TRN-002566_1001-03-T-CS...,GEOSP-TRN-002566_1001-03-T-CS-202-5600-C-DW-11...,None,None,None,None,None,...,TORRES_E48_A_E84_PV4,TORRES_E48_A_E84_PV4,100.000007,14,TORRES_E48_A_E84_PV4:100.00|TORRES_E48_A_E84_P...,False,<NA>,Fal

## 4. Exportar resultados

Usar `04_ready_for_datastore.csv` como entrada de la etapa de copia/carga, despues de revisar que `destination_path` este correcto.

In [7]:
summary_df = pd.DataFrame([
    {'metric': 'run_timestamp', 'value': run_timestamp},
    {'metric': 'audit_logic_version', 'value': AUDIT_LOGIC_VERSION},
    {'metric': 'input_folder', 'value': PATH_INPUT_IMAGENES},
    {'metric': 'sector_feature_class', 'value': PATH_FC_INDICE_VUELOS_IMGS},
    {'metric': 'datastore_destination_root', 'value': PATH_DATASTORE_DESTINO_RAIZ},
    {'metric': 'sector_field', 'value': SECTOR_FIELD_INDICE_VUELOS},
    {'metric': 'sector_query', 'value': QUERY_INDICE_VUELOS},
    {'metric': 'input_files_count', 'value': len(input_images_df)},
    {'metric': 'ortho_images_count', 'value': len(ortho_images_df)},
    {'metric': 'ready_for_datastore_count', 'value': len(ready_df)},
    {'metric': 'review_required_count', 'value': len(review_df)},
    {'metric': 'duplicate_original_expected_file_name_count', 'value': int(manifest_output_df['duplicate_expected_file_name'].sum())},
    {'metric': 'duplicate_expected_file_name_resolved_count', 'value': int(manifest_output_df['duplicate_was_resolved'].sum())},
])

for status, count in spatial_matches_df['spatial_status'].value_counts(dropna=False).items():
    summary_df.loc[len(summary_df)] = {'metric': f'spatial_status_{status}', 'value': int(count)}

for status, count in manifest_df['rename_status'].value_counts(dropna=False).items():
    summary_df.loc[len(summary_df)] = {'metric': f'rename_status_{status}', 'value': int(count)}

summary_csv = OUTPUT_DIR / '00_summary.csv'
input_csv = OUTPUT_DIR / '01_input_images.csv'
spatial_csv = OUTPUT_DIR / '02_spatial_matches.csv'
manifest_csv = OUTPUT_DIR / '03_manifest_paths_datastore.csv'
ready_csv = OUTPUT_DIR / '04_ready_for_datastore.csv'
review_csv = OUTPUT_DIR / '05_review_required.csv'

summary_df.to_csv(summary_csv, index=False, encoding='utf-8-sig')
input_images_df.to_csv(input_csv, index=False, encoding='utf-8-sig')
spatial_matches_df.to_csv(spatial_csv, index=False, encoding='utf-8-sig')
manifest_output_df.to_csv(manifest_csv, index=False, encoding='utf-8-sig')
ready_df.to_csv(ready_csv, index=False, encoding='utf-8-sig')
review_df.to_csv(review_csv, index=False, encoding='utf-8-sig')

display(summary_df)
print('Outputs exportados en:', OUTPUT_DIR)
print('Manifest completo:', manifest_csv)
print('CSV listo para etapa siguiente:', ready_csv)
print('CSV de revision:', review_csv)

,metric,value
0,run_timestamp,20260618_110837
1,audit_logic_version,2026-06-15-spatial-sector-preserve-token
2,input_folder,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...
3,sector_feature_class,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\...
4,datastore_destination_root,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO
5,sector_field,Sector
6,sector_query,Sensor <> 'DJI MATRICE 350 RTK'
7,input_files_count,607
8,ortho_images_count,33
9,ready_for_datastore_count,16


Outputs exportados en: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\flujo_geosupport_etapas\outputs\etapa_01_preparar_paths_datastore
Manifest completo: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\flujo_geosupport_etapas\outputs\etapa_01_preparar_paths_datastore\03_manifest_paths_datastore.csv
CSV listo para etapa siguiente: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\flujo_geosupport_etapas\outputs\etapa_01_preparar_paths_datastore\04_ready_for_datastore.csv
CSV de revision: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\flujo_geosupport_etapas\outputs\etapa_01_preparar_paths_datastore\05_review_required.csv
